# Part 1: Extract Training Data from Raster Files

This section extracts pixel values directly from yearly/quarterly raster datasets, applies band names, and performs basic cleaning (removing zeros and NaNs).


In [ ]:
import pandas as pd
import numpy as np
import rasterio
import os
import glob
import warnings

warnings.filterwarnings('ignore')


In [ ]:
def process_raster_only(raster_path, output_csv, col_names=None):
    print(f"Processing {raster_path}...")
    
    with rasterio.open(raster_path) as src:
        # Read all bands (shape will be: bands, rows, cols)
        data = src.read()
        
        # Flatten to (num_pixels, num_bands)
        num_bands = data.shape[0]
        pixels = data.reshape(num_bands, -1).T
        
        # Get valid pixels (remove 0 and NaN)
        valid_mask = ~np.all(pixels == 0, axis=1) & ~np.any(np.isnan(pixels), axis=1)
        valid_pixels = pixels[valid_mask]
        
        if valid_pixels.shape[0] == 0:
            print("No valid pixels found!")
            return
            
        print(f"Found {valid_pixels.shape[0]} valid pixels.")
        
        # Create DataFrame
        if col_names is None or len(col_names) != num_bands:
            col_names = [f"Band_{i+1}" for i in range(num_bands)]
            
        out_df = pd.DataFrame(valid_pixels, columns=col_names)
        
        # Final cleaning
        out_df.dropna(inplace=True)
        
        # Save to CSV
        out_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"Saved {len(out_df)} pixels to {output_csv}\n")

In [ ]:
# Descriptive band names
band_names = [
    'Blue', 'Green', 'Red', 'NIR', 'Thermal', 
    'Rain', 'Soil Moisture', 'Air Temp', 'ET', 
    'DEM', 'Soil Texture', 'Salinity_Index', 'VCI'
]

# Get all new TIFF files
tiff_files = glob.glob('TRAIN_DATA/*.tif')
tiff_files.sort()

print(f"Found {len(tiff_files)} TIFF files.")

for tiff in tiff_files:
    parts = os.path.basename(tiff).replace('.tif', '').split('_')
    if len(parts) >= 3:
        period = f"{parts[1]}_{parts[2]}"
    else:
        period = os.path.basename(tiff).split('.')[0]
        
    output_csv = f'train_data_{period}.csv'
    process_raster_only(tiff, output_csv, col_names=band_names)

# Part 2: Data Inspection

Verify the extracted data by loading a sample.


In [ ]:
import glob
import pandas as pd

# Load a sample from the first generated CSV to verify columns and data
csv_files = glob.glob('train_data_*.csv')
if csv_files:
    file_to_check = csv_files[0]
    print(f"Checking {file_to_check}...")
    df_sample = pd.read_csv(file_to_check, nrows=1000)
    
    print("\nData Info:")
    display(df_sample.info())
    print("\nFirst 5 rows:")
    display(df_sample.head())
else:
    print("No CSV files found. Please run Part 1 first.")